# 01 · Ingesta del DENUE y primer vistazo a los restaurantes

**Objetivo:** leer las 7 ediciones que bajó el notebook 00, limpiar los datos de la zona metropolitana y describir los **restaurantes por tipo**.

- **Todo el DENUE** (sin administración pública) se guarda como *contexto*: oficinas, escuelas o consultorios atraen clientes y serán variables del modelo.
- **Los restaurantes** (SCIAN 722, agrupados en 10 tipos) son lo que el sistema va a recomendar.
- La comparación que importa es **de censo a censo, 04/2020 → 05/2025**: entre censos el DENUE casi no registra cierres.

## 0. Librerías y rutas

In [ ]:
from pathlib import Path
import zipfile, io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import h3
import mapas                      # funciones de mapas compartidas (notebooks/mapas.py)

ROOT = Path.cwd().parent
RAW, PROC, FIG = ROOT / "data/raw", ROOT / "data/processed", ROOT / "figures"
for p in (PROC, FIG):
    p.mkdir(parents=True, exist_ok=True)
miles = FuncFormatter(lambda v, _: f"{v:,.0f}")

## 1. Configuración

- `EDICIONES_USAR`: los 7 cortes anuales (no hay 05/2023; se usa 11/2023).
- `EDICION_BASE` y `EDICION_FIN`: las dos ediciones con Censo Económico, entre las que sí se pueden medir entradas y salidas.
- `TIPOS_RESTAURANTE`: el catálogo de lo que se va a recomendar. Se excluyen los servicios de alimentos que no son un local abierto al público (comedores de empresa, banquetes, unidades móviles), y los centros nocturnos se juntan con los bares por su bajo número.

In [ ]:
EDICIONES_USAR = ["2020-04", "2021-05", "2022-05", "2023-11", "2024-05", "2025-05", "2026-05"]
EDICION_BASE, EDICION_FIN = "2020-04", "2025-05"

MUNICIPIOS_ZMQ = {6: "Corregidora", 8: "Huimilpan", 11: "El Marqués", 14: "Querétaro"}
LAT_MIN, LAT_MAX, LON_MIN, LON_MAX = 20.2, 21.0, -100.7, -99.9     # caja de la ZM
RES = 8                                                              # hexágonos de ~0.74 km²

TIPOS_RESTAURANTE = {
    "722511": "A la carta / comida corrida",
    "722512": "Mariscos",
    "722513": "Antojitos",
    "722514": "Tacos y tortas",
    "722515": "Cafeterías y neverías",
    "722516": "Autoservicio",
    "722517": "Pizzas, hamburguesas y pollos",
    "722518": "Otros para llevar",
    "722519": "Otros de consumo inmediato",
    "722411": "Bares y centros nocturnos",
    "722412": "Bares y centros nocturnos",
}

disponibles = {p.stem.replace("denue_qro_", ""): p.name for p in sorted(RAW.glob("denue_qro_*.parquet"))}
faltan = [e for e in EDICIONES_USAR if e not in disponibles]
if faltan:
    print("Aviso, faltan estas ediciones (córrelas en el notebook 00):", faltan)
EDICIONES = {e: disponibles[e] for e in EDICIONES_USAR if e in disponibles}
if not EDICIONES:
    raise FileNotFoundError(f"No hay ediciones en {RAW}. Corre primero 00_descarga_denue.ipynb.")
print(EDICIONES)

## 2. Función de lectura

Lee el Parquet que deja el notebook 00 y, por si descargaste a mano, también zip o csv. Pasa las columnas a minúsculas y lee los códigos como texto para no perder ceros a la izquierda.

In [ ]:
COLS = ["id", "nom_estab", "raz_social", "codigo_act", "nombre_act", "per_ocu",
        "cve_ent", "cve_mun", "municipio", "ageb", "manzana",
        "nom_vial", "numero_ext", "numero_int", "latitud", "longitud", "fecha_alta"]

def _normalizar(df):
    df.columns = df.columns.str.strip().str.lower()
    return df[[c for c in COLS if c in df.columns]]

def _leer_csv(buffer_bytes):
    for enc in ("utf-8", "latin-1"):
        try:
            return _normalizar(pd.read_csv(io.BytesIO(buffer_bytes), encoding=enc, dtype=str, low_memory=False))
        except UnicodeDecodeError:
            continue

def leer_denue(ruta: Path) -> pd.DataFrame:
    if ruta.suffix.lower() == ".parquet":
        return _normalizar(pd.read_parquet(ruta))
    if ruta.suffix.lower() == ".zip":
        with zipfile.ZipFile(ruta) as z:
            mayor = max((i for i in z.infolist() if i.filename.lower().endswith(".csv")), key=lambda i: i.file_size)
            return _leer_csv(z.read(mayor))
    return _leer_csv(ruta.read_bytes())

## 3. Limpiar y guardar (todo el DENUE, como contexto)

Por edición:

1. Filtra los 4 municipios y quita administración pública (SCIAN 93).
2. Descarta coordenadas fuera de la zona metropolitana (errores de georreferencia).
3. Guarda `data/processed/denue_zmq_AAAA-MM.parquet` y una bitácora de los registros descartados.

In [ ]:
def preparar(df, etiqueta):
    df = df.copy()
    df["cve_ent"] = pd.to_numeric(df["cve_ent"], errors="coerce")
    df["cve_mun"] = pd.to_numeric(df["cve_mun"], errors="coerce")
    df = df[(df["cve_ent"] == 22) & (df["cve_mun"].isin(MUNICIPIOS_ZMQ))]
    df["codigo_act"] = df["codigo_act"].str.strip().str.zfill(6)
    df = df[~df["codigo_act"].str.startswith("93")]
    df["latitud"] = pd.to_numeric(df["latitud"], errors="coerce")
    df["longitud"] = pd.to_numeric(df["longitud"], errors="coerce")

    dentro = df["latitud"].between(LAT_MIN, LAT_MAX) & df["longitud"].between(LON_MIN, LON_MAX)
    descartados = df.loc[~dentro, ["id", "nom_estab", "municipio", "latitud", "longitud"]].assign(edicion=etiqueta)
    if len(descartados):
        print(f"  {etiqueta}: {len(descartados)} registros con coordenadas fuera de la ZM, descartados")
    df = df[dentro]

    df["edicion"] = etiqueta
    df["mun_nombre"] = df["cve_mun"].map(MUNICIPIOS_ZMQ)
    return df, descartados

frames, fuera = [], []
for etiqueta, archivo in sorted(EDICIONES.items()):
    print("Leyendo", etiqueta)
    df, desc = preparar(leer_denue(RAW / archivo), etiqueta)
    df.to_parquet(PROC / f"denue_zmq_{etiqueta}.parquet", index=False)
    frames.append(df)
    fuera.append(desc)

denue = pd.concat(frames, ignore_index=True)
pd.concat(fuera, ignore_index=True).to_csv(PROC / "registros_fuera_de_zm.csv", index=False)
denue.groupby("edicion").size().rename("establecimientos")

## 4. Calidad básica por edición

In [ ]:
calidad = denue.groupby("edicion").agg(
    registros=("id", "size"),
    ids_unicos=("id", "nunique"),
    fecha_alta_max=("fecha_alta", "max"),
)
calidad

## 5. Restaurantes por tipo

Se quedan solo los códigos del catálogo. El resultado, con todas las ediciones juntas, se guarda en `data/processed/restaurantes_zmq.parquet` y es la base de los notebooks siguientes.

In [ ]:
rest = denue[denue["codigo_act"].isin(TIPOS_RESTAURANTE)].copy()
rest["tipo"] = rest["codigo_act"].map(TIPOS_RESTAURANTE)
rest.to_parquet(PROC / "restaurantes_zmq.parquet", index=False)

por_tipo = rest.groupby(["tipo", "edicion"]).size().unstack(fill_value=0)
por_tipo = por_tipo.loc[por_tipo[EDICION_BASE].sort_values(ascending=False).index]
print(f"Restaurantes en {EDICION_BASE}: {por_tipo[EDICION_BASE].sum():,} · en {EDICION_FIN}: {por_tipo[EDICION_FIN].sum():,}")
por_tipo

## 6. Gráfica 1 · Restaurantes por edición

La línea punteada marca dónde entra el Censo Económico 2024: el brinco de esa edición es sobre todo actualización del directorio, no aperturas de un año.

In [ ]:
tabla = (rest.groupby(["edicion", "mun_nombre"]).size().unstack(fill_value=0)
             [["Querétaro", "El Marqués", "Corregidora", "Huimilpan"]])
tot = tabla.sum(axis=1)

fig, ax = plt.subplots(figsize=(11, 6))
tabla.plot(kind="bar", stacked=True, ax=ax, width=0.6, color=["#14213D", "#2E6F95", "#E8A25F", "#C9C2B4"])
ax.set_ylim(0, tot.max() * 1.18); ax.yaxis.set_major_formatter(miles)
ax.set_xlabel("Edición del DENUE"); ax.set_ylabel("Restaurantes")
ax.set_title("Restaurantes en la ZM de Querétaro", loc="left")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(title="Municipio", loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False)
for i, t in enumerate(tot):
    ax.text(i, t + tot.max() * 0.01, f"{t:,}", ha="center", va="bottom", fontsize=11)
if {"2024-05", "2025-05"} <= set(tabla.index):
    x = list(tabla.index).index("2024-05") + 0.5
    ax.axvline(x, color="#A8551A", linestyle="--", linewidth=1.5)
    ax.text(x - 0.08, tot.max() * 1.13, "Entra el Censo Económico 2024 →",
            color="#A8551A", fontsize=11, ha="right", va="center")
plt.xticks(rotation=0); plt.tight_layout()
plt.savefig(FIG / "01_restaurantes_por_edicion.png", dpi=200, bbox_inches="tight"); plt.show()

## 7. Gráfica 2 · Tipos de restaurante, de censo a censo

Compara cuántos había de cada tipo en 04/2020 y en 05/2025. Es cambio neto: las entradas y salidas por separado se construyen en el notebook 03.

In [ ]:
comp = por_tipo[[EDICION_BASE, EDICION_FIN]].iloc[::-1]
cambio = comp[EDICION_FIN] / comp[EDICION_BASE] - 1

fig, ax = plt.subplots(figsize=(11, 7))
y = np.arange(len(comp)); h = 0.38
ax.barh(y + h / 2, comp[EDICION_BASE], h, color="#C9C2B4", label=EDICION_BASE)
ax.barh(y - h / 2, comp[EDICION_FIN], h, color="#2E6F95", label=EDICION_FIN)
for yi, v, c in zip(y, comp[EDICION_FIN], cambio):
    ax.text(v + comp.values.max() * 0.01, yi - h / 2, f"{v:,}  ({c:+.0%})", va="center", fontsize=10)
ax.set_yticks(y, comp.index); ax.xaxis.set_major_formatter(miles)
ax.set_xlim(0, comp.values.max() * 1.25); ax.set_xlabel("Restaurantes")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="lower right")
ax.set_title(f"Restaurantes por tipo · {EDICION_BASE} vs {EDICION_FIN}", loc="left")
plt.tight_layout(); plt.savefig(FIG / "02_restaurantes_por_tipo.png", dpi=200, bbox_inches="tight"); plt.show()

## 8. Gráfica 3 · Mapa de densidad de restaurantes

Hexágonos H3 de resolución 8 sobre el mapa base gris, con clases en lugar de escala continua. Se usa la edición `EDICION_FIN`.

In [ ]:
fin = rest[rest["edicion"] == EDICION_FIN]
fin = fin.assign(h3=[h3.latlng_to_cell(la, lo, RES) for la, lo in zip(fin["latitud"], fin["longitud"])])
conteo = fin.groupby("h3").size()

BORDES = [1, 2, 5, 10, 25, 50, conteo.max() + 1]
ETIQ = ["1", "2–4", "5–9", "10–24", "25–49", f"50–{conteo.max():,}"]
cmap, norm = mapas.escala_por_clases(BORDES, ["#FFF3D6", "#FDD49E", "#FDAE6B", "#F16913", "#C2410C", "#7F2704"])

fig, ax, polys = mapas.base_mapa(conteo.index)
coll = mapas.capa_hex(ax, polys, array=conteo.values, cmap=cmap, norm=norm)
mapas.referencias(ax)
mapas.barra_clases(fig, ax, coll, BORDES, ETIQ, "Restaurantes por hexágono (~0.74 km²)")
ax.set_title(f"Densidad de restaurantes en la ZM de Querétaro · {EDICION_FIN}", loc="left", fontsize=13)
plt.tight_layout(); plt.savefig(FIG / "03_mapa_restaurantes.png", dpi=200, bbox_inches="tight"); plt.show()
print(f"Hexágonos con al menos un restaurante: {len(conteo):,} · mediana: {conteo.median():.0f} · máximo: {conteo.max():,}")

### 8b. (Opcional) Mapa interactivo

In [ ]:
# import folium
# from matplotlib.colors import to_hex
# m = folium.Map(location=[20.6, -100.4], zoom_start=11,
#                tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}",
#                attr="Esri, HERE, Garmin, OpenStreetMap contributors")
# for c, n in conteo.items():
#     folium.Polygon(h3.cell_to_boundary(c), weight=0, fill=True, fill_opacity=0.7,
#                    fill_color=to_hex(cmap(norm(n))), tooltip=f"{n:,} restaurantes").add_to(m)
# m.save(FIG / "03_mapa_restaurantes_interactivo.html"); m

## 9. Números para la presentación

In [ ]:
b, f = por_tipo[EDICION_BASE].sum(), por_tipo[EDICION_FIN].sum()
print(f"Restaurantes {EDICION_BASE}: {b:,} · {EDICION_FIN}: {f:,} · cambio neto: {f - b:+,} ({f / b - 1:+.1%})")
print(f"Tipos de restaurante: {len(por_tipo)}")
print("Mayor crecimiento:", cambio.idxmax(), f"({cambio.max():+.0%})")
print("Menor crecimiento:", cambio.idxmin(), f"({cambio.min():+.0%})")